In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits
from sklearn.cluster import KMeans
import sys
import os
import lightkurve as lk
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parentsfit_M64.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
#from hogg
foo, k = squared_feats.shape
print(squared_feats.shape)
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)
refined_freqs = data["refined_frequency"]

In [ ]:
features = data["features"]
print(features.shape)
kics = data["star_id"]
print(kics.shape)

#create the time invariant feature vector for clustering (log(a0^2), log(a1^2 + b1^2), ...)
squared_feats = np.zeros((1431, 65))
for e,f in enumerate(features):
    j = 0
    for i in range(len(f) - 1):
        if i == 0:
            squared_feats[e][j] = f[i]**2
            j = j + 1
        if i%2 == 1:
            squared_feats[e][j] = f[i]**2 + f[i+1]**2
            j = j + 1

foo, k = squared_feats.shape
print(squared_feats.shape)
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)


refined_freqs = data["refined_frequency"]
good_mask = ~np.any(np.isnan(squared_feats), axis=1)
print("good mask", good_mask.shape)
print("squared feats before", squared_feats.shape)
squared_feats = squared_feats[good_mask]

print(squared_feats.shape)
log_squared_feats = np.log10(squared_feats)
log_squared_feats = log_squared_feats[:, 1:] #get rid of constant?

syms = ["o", "x", "s", "D", "+", "*", "p", "^", "1"]
sizes = [3, 7, 11, 15, 18, 21]

kics = kics[good_mask]
refined_freqs = refined_freqs[good_mask]
features = features[good_mask]
print(kics.shape, refined_freqs.shape)
print(np.all(squared_feats))

In [ ]:
def phase_folding_at_freq(group, k, random_restart, kmeans):
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    
    mask = labels == group
    
    
    true_indices = np.where(mask)[0] #get indices where its true in mask
    #i = true_indices[2]#choose the first star indec
    dists = np.linalg.norm(log_squared_feats[true_indices] - centroids[group], axis=1)
    i = true_indices[np.argmin(dists)] #takes index from label array to find corresponding 1400 index
    print(i)
    star_id, feature, freq = kics[i], features[i], refined_freqs[i]

    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
   
    over_sampling = 3
    df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)

    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()

    plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    # plt.axvline(fc)
    # plt.axvline(fc/2)
    plt.xlabel("Frequency (1/day)")
    plt.ylabel("Power")
    plt.semilogy()
    plt.title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        plt.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        plt.axvline(cf, color='red', alpha=0.2, lw=0.5)
    plt.show()
    
    print("feature[0]", feature[0])
    print("feature length", (feature[0]).shape)
          
    theta = np.linspace(0, 4*np.pi, 1000)
    yplot0 = np.zeros_like(theta)

    for m in range(1, 65):
        a = feature[2*m-1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
        
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)
    yplot = flux_fit - np.nanmean(flux_fit)
    #yplot0 -= np.nanmean(yplot0)

    plt.figure()
    plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    plt.title(f'phase fold @ freq, K = {k}, group = {group}, freq= {freq:.4f}')
    plt.show()



## By closest to centroid

In [ ]:
K, r = 10, 33

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
kmeans.fit(log_squared_feats)
for group in range(K):
    phase_folding_at_freq(group, K, 33, kmeans)

In [ ]:
def phase_folding_at_freq_next(group, k, random_restart, kmeans, rank=0):
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    
    mask = labels == group
    true_indices = np.where(mask)[0]
    dists = np.linalg.norm(log_squared_feats[true_indices] - centroids[group], axis=1)
    sorted_indices = true_indices[np.argsort(dists)]
    if rank >= len(sorted_indices):
        print(f"rank {rank} out of range, only {len(sorted_indices)} stars in group")
        return
    i = sorted_indices[rank]
    
    star_id, feature, freq = kics[i], features[i], refined_freqs[i]
    print(f"rank: {rank}, index: {i}, star_id: {star_id}, freq: {freq:.4f}")

    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
   
    over_sampling = 3
    df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()
    plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    plt.xlabel("Frequency (1/day)")
    plt.ylabel("Power")
    plt.semilogy()
    plt.title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        plt.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        plt.axvline(cf, color='red', alpha=0.2, lw=0.5)
    plt.show()

    M = (len(feature) - 1) // 2
    print(f"feature length: {len(feature)}, M: {M}")

    theta = np.linspace(0, 4*np.pi, 1000)
    yplot0 = feature[0]
    for m in range(1, M + 1):
        a = feature[2*m - 1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    yplot0 -= np.nanmean(yplot0)
        
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)
    yplot = flux_fit - np.nanmean(flux_fit)
    plt.figure()
    plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    plt.title(f'phase fold @ freq, K = {k}, group = {group}, rank = {rank}, freq= {freq:.4f}')
    plt.show()

In [ ]:
K, r = 10, 33

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
kmeans.fit(log_squared_feats)
for group in range(K):
    phase_folding_at_freq_next(group, K, 33, kmeans, 0)